# Step 7: 量化方法时间线 + 为何 PTQ 为主

**目标**：梳理 LLM 量化方法的历史演进（LLM.int8() → GPTQ → SmoothQuant → AWQ → FP8 → …），并理解为何**后训练量化（PTQ）**是主流、训练量化（QAT）何时才值得。**纯 CPU 概念 step**。

**对应 OUTLINE 课时**：1.7 时间线 + PTQ 为主（~30 分钟）。

> 本 step 无 GPU 代码（概念 step 例外，同 s6）。

In [ ]:
%%capture
import math, json, pathlib
import torch
import torch.nn as nn
import ipytest
ipytest.autoconfig()

In [ ]:
# Setup cell：notebook 向上发现模块根（含 steps/ + pyproject.toml），绝不依赖裸相对路径。
# 规范见 course/NOTEBOOK_CONVENTIONS.md 第 2 节。所有文件路径从 MODULE_ROOT 派生。
def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "steps").is_dir() and (cand / "pyproject.toml").exists():
            return cand
    raise RuntimeError("找不到模块根（含 steps/ + pyproject.toml）；请在模块目录内 cd course/m1-activation-outliers 启动 jupyter")

MODULE_ROOT    = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"      # 与 scripts/download_model.sh 一致
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"     # L3 先在 0.5B 上验，再上 7B
OUT_ROOT       = MODULE_ROOT / "out"                                 # 已 gitignore
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT:", MODULE_ROOT)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("GPU:", torch.cuda.get_device_name(), "（sm{}{}，cap={}）".format(cap[0], cap[1], cap),
          "— 支持 FP8" if cap >= (8, 9) else "")
else:
    print("无 GPU（仅 L1/L2 可跑；L3 自动跳过）")

## 原理：方法演进时间线

LLM 量化方法不是凭空出现的，每个都是对前一个痛点的回应：

| 年份 | 方法 | 解决的痛点 | 核心思想 |
|------|------|-----------|----------|
| 2022.08 | **LLM.int8()** | emergent outlier 击垮量化（s1/s2） | 混合精度分解：outlier 切 FP16 相加 |
| 2022.10 | **GPTQ** | 朴素 per-tensor INT4 精度崩 | 基于 Hessian 信息的逐层权重补偿（weight-only W4A16，arXiv 2210.17323） |
| 2022.11 | **SmoothQuant** | LLM.int8() 双路径慢 | 等价缩放迁移：outlier 搬进权重，全 INT8（s3，arXiv 2211.10438） |
| 2023.06 | **AWQ** | GPTQ 需校准数据、对显著通道粗糙 | 激活感知的显著通道权重缩放（weight-only W4A16，s4） |
| 2022-2023 | **FP8 (硬件)** | INT8 抗离群仍需校准 | Hopper/Ada 原生 FP8，动态范围抗离群、无需校准（s5；H100 架构 2022.03 发布，2023 起规模交付） |

**演进主线**：从"粗暴 per-tensor"→"识别并处理 outlier"（LLM.int8/SmoothQuant/AWQ）→"硬件原生低精度"（FP8）。每一步都在降低"量化精度损失"与"工程复杂度"的乘积。

### PTQ vs QAT

| | PTQ（后训练量化） | QAT（量化感知训练） |
|---|---|---|
| **流程** | 拿到训练好的 FP16 模型，**不重训**，用少量校准数据调 scale | 把量化算子插入训练图，**继续训练**让权重适应量化 |
| **成本** | 分钟-小时（校准 128-512 条样本即可） | 天-周（需完整训练基础设施、数据） |
| **精度** | INT8/FP8 接近无损；INT4 轻微掉点（1-2%） | INT4/INT2 也能接近无损（极限场景） |
| **何时用** | **99% 场景**：部署前快速量化 | 极低位宽（INT2/INT1）、PTQ 掉点超容忍 |

**为何 PTQ 为主**：LLM 训练成本极高（7B 训练需数百 GPU-天、海量数据、对齐流程）。QAT 要复刻这套基础设施只为省几个点精度，性价比极低。而现代 PTQ（SmoothQuant/AWQ/FP8）在 INT8/INT4 已逼近无损——**除非要做极限低比特（<4bit），否则 PTQ 足够**。

## 本步填空

1. **`method_meta(method)`** —— 返回某方法的元数据 dict（年份/痛点/核心思想）。
2. **`ptq_vs_qat(n_params, target_bits, has_training_infra, budget_days)`** —— **判断型**：按规模/目标位宽/预算推荐 PTQ 还是 QAT。

In [ ]:
def method_meta(method):
    """返回某量化方法的元数据。

    参数
    ----
    method : str，之一："gptq" / "llmint8" / "smoothquant" / "awq" / "fp8"。

    返回
    ----
    dict，含键：
      - "name"   : str  方法全名
      - "year"   : int  发表年份（如 2022）
      - "solves" : str  解决的痛点（一句话）
      - "idea"   : str  核心思想（一句话）
      - "module" : str  对应本课程哪里：
                        * llmint8→"s2"、smoothquant→"s3"、awq→"s4"、fp8→"s5"（本模块 M1 有 step）；
                        * gptq→"M2"（GPTQ 的工程实现在 M2 llm-compressor 工具链，M1 只在时间线提及）。

    数据见原理 cell 的时间线表（不要凭记忆瞎编年份，对照表格）。
    """
    # TODO: 返回方法元数据 dict。
    raise NotImplementedError


def ptq_vs_qat(n_params, target_bits, has_training_infra=False, budget_days=1):
    """按约束推荐 PTQ 还是 QAT（判断型核心）。

    参数
    ----
    n_params : int/float，模型规模（如 7e9）。
    target_bits : int，目标权重位宽（4/8/2…）。
    has_training_infra : bool，是否有完整训练基础设施（数据/对齐/算力）。
    budget_days : float，可接受的量化工期（天）。PTQ 典型 <1 天；QAT 需 数天-数周。

    返回
    ----
    str："ptq" 或 "qat"。

    决策规则（对齐原理）：
      1. target_bits >= 4 → "ptq"（现代 PTQ 在 INT4/INT8 已近无损）。
      2. target_bits < 4（极限低比特）：
         - has_training_infra 且 budget_days >= 7 → "qat"（值得投入，QAT 能救回精度）。
         - 否则 → "ptq"（没条件 QAT，只能 PTQ + 接受掉点，或放弃该位宽）。
      3. 无论 target_bits，若 budget_days < 1 且无 infra → "ptq"（PTQ 是默认）。

    提示（判断训练点）：
      - 想想为什么 target_bits 是第一判断维度——位宽越低，PTQ 的精度损失越陡，
        到 INT2 以下 PTQ 基本不可用，必须 QAT。但 INT4+ PTQ 足够。
      - QAT 的门槛是"infra + 时间"，缺一不可。
    """
    # TODO: 实现 PTQ vs QAT 决策。
    raise NotImplementedError

In [ ]:
%%ipytest -qq

def test_method_meta_keys_and_values():
    m = method_meta("smoothquant")
    assert set(m.keys()) >= {"name", "year", "solves", "idea", "module"}
    assert m["year"] == 2022
    assert m["module"] == "s3"
    assert "outlier" in m["idea"].lower() or "迁移" in m["idea"]

def test_method_meta_fp8_no_calibration():
    m = method_meta("fp8")
    assert m["year"] >= 2023
    assert m["module"] == "s5"

def test_method_meta_all_known_and_traced():
    # 每个 method 的 module 要么指向本模块某 step（s1..s7），要么显式标注跨模块（"M2"）。
    # GPTQ 的工程实现在 M2（llm-compressor 工具链），M1 只在时间线里作为演进背景提及——
    # 故 GPTQ 的 module 标 "M2"，其余四个方法（llmint8/smoothquant/awq/fp8）在本模块有 step。
    for meth in ["gptq", "llmint8", "smoothquant", "awq", "fp8"]:
        m = method_meta(meth)
        assert isinstance(m["year"], int) and 2020 <= m["year"] <= 2025
        mod = m["module"]
        assert mod.startswith("s") or mod == "M2", f"{meth} 的 module 应是 sN 或 'M2'，实际 {mod!r}"
    # M1 有 step 的四个方法：
    assert method_meta("llmint8")["module"].startswith("s")
    assert method_meta("smoothquant")["module"].startswith("s")
    assert method_meta("awq")["module"].startswith("s")
    assert method_meta("fp8")["module"].startswith("s")
    # GPTQ 跨模块（M2 工具链）：
    assert method_meta("gptq")["module"] == "M2"

def test_ptq_vs_qat_high_bits_always_ptq():
    assert ptq_vs_qat(7e9, target_bits=4) == "ptq"
    assert ptq_vs_qat(7e9, target_bits=8, has_training_infra=True, budget_days=30) == "ptq"

def test_ptq_vs_qat_low_bits_needs_qat_when_capable():
    # INT2 极限低比特 + 有 infra + 有时间 → qat
    assert ptq_vs_qat(7e9, target_bits=2, has_training_infra=True, budget_days=14) == "qat"

def test_ptq_vs_qat_low_bits_falls_back_without_infra():
    # INT2 但没训练 infra → 只能 ptq（接受掉点）
    assert ptq_vs_qat(7e9, target_bits=2, has_training_infra=False, budget_days=30) == "ptq"

def test_ptq_vs_qat_low_bits_no_time_falls_back():
    # INT2 有 infra 但没时间（budget<7天）→ ptq
    assert ptq_vs_qat(7e9, target_bits=2, has_training_infra=True, budget_days=3) == "ptq"

## L2：tiny 时间线渲染 + PTQ/QAT 自洽（CPU）

把方法时间线渲染成可读表，验证 PTQ/QAT 决策在不同规模/位宽下自洽。

In [ ]:
# 时间线渲染
methods = ["gptq", "llmint8", "smoothquant", "awq", "fp8"]
print("=== LLM 量化方法时间线 ===")
print(f"{'年份':6s} {'方法':14s} {'解决痛点':30s} {'本课step'}")
rows = []
for m in sorted(methods, key=lambda x: method_meta(x)["year"]):
    meta = method_meta(m)
    rows.append(meta)
    print(f"{meta['year']:6d} {meta['name']:14s} {meta['solves'][:30]:30s} {meta['module']}")
print("\nL2a PASS：时间线渲染")

# PTQ/QAT 决策矩阵自洽
print("\n=== PTQ vs QAT 决策矩阵 ===")
cases = [
    (7e9, 8, False, 0.5),     # INT8，快速 → ptq
    (7e9, 4, False, 1),       # INT4 → ptq
    (7e9, 2, True, 14),       # INT2 全副武装 → qat
    (7e9, 2, False, 30),      # INT2 无 infra → ptq
    (70e9, 2, True, 3),       # INT2 有 infra 没时间 → ptq
]
for n, bits, infra, days in cases:
    rec = ptq_vs_qat(n, bits, infra, days)
    print(f"  {n/1e9:.0f}B INT{bits} infra={infra} budget={days}d -> {rec.upper()}")
# 自洽：所有 INT4+ 必 ptq
assert all(ptq_vs_qat(7e9, b, True, 30) == "ptq" for b in [4, 8])
print("L2b PASS：PTQ/QAT 决策矩阵自洽（INT4+ 恒 PTQ）")

## L3：方法表（CPU 概念，无 GPU）

汇总全模块方法元数据 + 选型关联，写 `method_table.json`。**无 GPU 代码**。

In [ ]:
table = []
for m in ["gptq", "llmint8", "smoothquant", "awq", "fp8"]:
    meta = method_meta(m)
    table.append({
        "key": m, "name": meta["name"], "year": meta["year"],
        "solves": meta["solves"], "idea": meta["idea"], "module": meta["module"],
        "is_ptq": True,   # 本课讲的全部是 PTQ
    })
(OUT_ROOT/"method_table.json").write_text(json.dumps(table, indent=2, ensure_ascii=False))

# PTQ 为主结论
n_ptq = sum(1 for t in table if t["is_ptq"])
print(f"本模块覆盖 {len(table)} 个方法，其中 {n_ptq} 个是 PTQ。")
print("→ 印证：现代 LLM 量化的主流是 PTQ（部署前快速量化，无需重训）。")
print("\nL3 PASS：method_table.json 写入", OUT_ROOT/"method_table.json")

## 产物检查

打印 `method_table.json` 全表，回顾本模块方法地图。

In [ ]:
def report_methods():
    path = OUT_ROOT / "method_table.json"
    table = json.loads(path.read_text())
    print("=== M1 方法地图（按时间）===")
    for t in sorted(table, key=lambda x: x["year"]):
        loc = t['module'] if t['module'].startswith("s") else f"{t['module']}（工具链）"
        print(f"\n[{t['year']}] {t['name']} ({t['key']}) — 见 {loc}")
        print(f"  痛点: {t['solves']}")
        print(f"  思想: {t['idea']}")
    print(f"\n全 {len(table)} 方法均为 PTQ —— 现代部署的默认选择。")
    # 完整性：5 个方法，每个要么链到本模块 step（sN），要么显式跨模块标注（"M2"）。
    assert len(table) == 5
    assert all(t["module"].startswith("s") or t["module"] == "M2" for t in table)
    # GPTQ 跨模块（工程实现在 M2），其余四个在本模块有 step——与原理表的时间线一致。
    n_in_m1 = sum(1 for t in table if t["module"].startswith("s"))
    assert n_in_m1 == 4, f"应有 4 个方法链到 M1 step，实际 {n_in_m1}"
    print(f"\n✓ 方法表完整（5 方法，全 PTQ；{n_in_m1} 个链到本模块 step，GPTQ 标注 M2 工具链）")

report_methods()